# VAR — Vector Autoregression (Turkey, Pipeline B)

**Model**: `statsmodels.tsa.api.VAR` / **Target**: `ngdprsaxdctrq`  
**Variables**: Cat2 — ipi_sa, usd_try_avg, cpi_sa, fin_acc (COVID dummies excluded from VAR system — constant in training)  
**Train**: 1995-Q2 to 2011-Q4 / **Val**: 2012-Q1 to 2017-Q4 / **Test**: 2018-Q1 to 2025-Q4 (32 quarters)  
**Scaling**: NO / **HP tuning**: NO / **n_lags**: 3 (flatten), **VAR_LAGS**: 1 (endogenous lag order)  
**Key fix**: Subset to Cat2+target cols before flattening to prevent unrelated late-starting series causing empty dropna.


In [1]:
import sys, os
import numpy as np
import pandas as pd
from statsmodels.tsa.api import VAR as SM_VAR

sys.path.insert(0, os.path.join(os.path.pardir, 'turkey_data'))
from turkey_helpers import (
    gen_lagged_data, gen_vintage_data, flatten_data, mean_fill_dataset,
    get_features, load_data,
    PREDICTIONS_DIR, TARGET, COVID
)

SEED = 42
np.random.seed(SEED)

TRAIN_START = '1995-01-01'
TRAIN_END   = '2011-12-31'
VAL_END     = '2017-12-31'
TEST_START  = '2018-01-01'
TEST_END    = '2025-12-31'
N_LAGS      = 3   # flatten_data lags
VAR_LAGS    = 1   # endogenous VAR lag order
VINTAGES    = {'m1': -2, 'm2': -1, 'm3': 0, 'post1': 1, 'post2': 2}

print('VAR (Turkey) — Cat2, n_lags={}, VAR_LAGS={}, backend=statsmodels'.format(N_LAGS, VAR_LAGS))


VAR (Turkey) — Cat2, n_lags=3, VAR_LAGS=1, backend=statsmodels


In [2]:
monthly, _, metadata = load_data()
# Cat2 WITHOUT covid (COVID dummies are constant in training window → excluded from VAR)
features_no_covid = get_features('cat2', with_covid=False)
features_all      = get_features('cat2', with_covid=True)
print('Cat2 endogenous vars: {}'.format(features_no_covid))

# KEY FIX: Subset monthly to only target + Cat2 vars before pre-computation
# This prevents late-starting Turkey series causing empty dropna in the full monthly df
use_base_cols = ['date', TARGET] + [c for c in features_no_covid if c in monthly.columns]
monthly_sub   = monthly[use_base_cols].copy()

# Pre-compute which flattened columns are constant in training window (for column selection)
pre_train        = monthly_sub[(monthly_sub['date'] >= TRAIN_START) & (monthly_sub['date'] <= TRAIN_END)].reset_index(drop=True)
# Drop any entirely-NaN columns
all_nan_cols = [c for c in pre_train.columns if c != 'date' and pre_train[c].notna().sum() == 0]
pre_train    = pre_train.drop(columns=all_nan_cols)
pre_train_filled = mean_fill_dataset(pre_train, pre_train)
pre_train_flat   = flatten_data(pre_train_filled, TARGET, N_LAGS)
pre_train_flat   = pre_train_flat.loc[pre_train_flat.date.dt.month.isin([3, 6, 9, 12]), :]
pre_train_flat   = pre_train_flat.dropna(axis=0, how='any').reset_index(drop=True)

all_cols  = [c for c in pre_train_flat.columns if c != 'date']
cat2_cols = [
    c for c in all_cols
    if any(c == f or c.startswith(f + '_') for f in features_no_covid) and c != TARGET
]
nunique      = pre_train_flat[cat2_cols].nunique()
cols_to_drop = nunique[nunique <= 1].index.tolist()
cat2_cols    = [c for c in cat2_cols if c not in cols_to_drop]

print('Training quarters available: {}'.format(len(pre_train_flat)))
print('Cat2 flattened cols: {} used, {} dropped (constant in training)'.format(
    len(cat2_cols), len(cols_to_drop)))
if cols_to_drop:
    print('  Dropped: {}'.format(cols_to_drop))


Cat2 endogenous vars: ['ipi_sa', 'usd_try_avg', 'cpi_sa', 'fin_acc']
Training quarters available: 67
Cat2 flattened cols: 16 used, 0 dropped (constant in training)


In [3]:
# Rolling test loop
test_quarters = monthly[
    (monthly['date'] >= TEST_START) &
    (monthly['date'] <= TEST_END) &
    monthly['date'].dt.month.isin([3, 6, 9, 12])
]['date'].tolist()

predictions  = {v: [] for v in VINTAGES}
actuals_list = []
failed = 0

for i, q_date in enumerate(test_quarters):
    pd_q = pd.Timestamp(q_date)
    actual = monthly[monthly['date'] == pd_q][TARGET].values[0]
    actuals_list.append(actual)

    base = monthly_sub[(monthly_sub['date'] >= TRAIN_START) & (monthly_sub['date'] <= pd_q)].reset_index(drop=True)

    for vn, offset in VINTAGES.items():
        vintage_date = pd_q + pd.DateOffset(months=offset)
        train_m = gen_vintage_data(metadata, base.copy(), pd_q, vintage_date)

        # Shift target by 3 months so one-step VAR forecast maps to target-quarter GDP.
        train_m[TARGET] = train_m[TARGET].shift(3)

        train_f  = mean_fill_dataset(train_m, train_m)
        train_fl = flatten_data(train_f, TARGET, N_LAGS)
        train_fl = train_fl.loc[train_fl.date.dt.month.isin([3, 6, 9, 12]), :]
        train_fl = train_fl.dropna(axis=0, how='any').reset_index(drop=True)

        feature_cols = [c for c in cat2_cols if c in train_fl.columns]

        if len(train_fl) < 10 or len(feature_cols) < 1:
            predictions[vn].append(np.nan)
            continue

        try:
            var_cols = [TARGET] + feature_cols
            var_data = train_fl[var_cols].values

            model   = SM_VAR(var_data)
            results = model.fit(VAR_LAGS, trend='c')
            fcst    = results.forecast(var_data[-VAR_LAGS:], steps=1)
            pred    = fcst[0, 0]
        except Exception:
            pred = np.nan
            failed += 1

        predictions[vn].append(pred)

    if (i + 1) % 8 == 0:
        print('  {} / {}'.format(i + 1, len(test_quarters)))

print('Done. {} quarters, {} failures.'.format(len(test_quarters), failed))


  8 / 32


  16 / 32


  24 / 32


  32 / 32
Done. 32 quarters, 0 failures.


In [4]:
os.makedirs(PREDICTIONS_DIR, exist_ok=True)
for vn in VINTAGES:
    df = pd.DataFrame({'date': test_quarters, 'actual': actuals_list, 'prediction': predictions[vn]})
    path = os.path.join(PREDICTIONS_DIR, 'var_{}.csv'.format(vn))
    df.to_csv(path, index=False)
    print('Saved {} ({} rows) pred=[{:.4f},{:.4f}]'.format(
        os.path.basename(path), len(df), df['prediction'].min(), df['prediction'].max()))


Saved var_m1.csv (32 rows) pred=[-0.0051,0.0339]
Saved var_m2.csv (32 rows) pred=[-0.0373,0.0629]
Saved var_m3.csv (32 rows) pred=[-0.2143,0.1458]
Saved var_post1.csv (32 rows) pred=[-0.1615,0.1544]
Saved var_post2.csv (32 rows) pred=[-0.1048,0.1620]


In [5]:
def rmse(a, p):
    m = ~(np.isnan(a) | np.isnan(p))
    return np.sqrt(np.mean((a[m]-p[m])**2)) if m.sum()>0 else np.nan

def mae(a, p):
    m = ~(np.isnan(a) | np.isnan(p))
    return np.mean(np.abs(a[m]-p[m])) if m.sum()>0 else np.nan

panels = {
    'Pre-crisis (2018-2019)': ('2018-01-01', '2019-12-31'),
    'COVID      (2020-2021)': ('2020-04-01', '2021-12-31'),
    'Post-COVID (2022-2025)': ('2022-01-01', '2025-12-31'),
    'Full test  (2018-2025)': ('2018-01-01', '2025-12-31'),
}
a = np.array(actuals_list); d = pd.to_datetime(test_quarters)
print('VAR (Turkey) — Evaluation by Panel & Vintage')
for pn, (ps, pe) in panels.items():
    m = (d >= ps) & (d <= pe)
    print('--- {} ---'.format(pn))
    for vn in VINTAGES:
        pp = np.array(predictions[vn])
        print('  {}  RMSFE={:.6f}  MAE={:.6f}  N={}'.format(vn, rmse(a[m],pp[m]), mae(a[m],pp[m]), m.sum()))


VAR (Turkey) — Evaluation by Panel & Vintage
--- Pre-crisis (2018-2019) ---
  m1  RMSFE=0.014039  MAE=0.011697  N=8
  m2  RMSFE=0.010655  MAE=0.009927  N=8
  m3  RMSFE=0.016656  MAE=0.012027  N=8
  post1  RMSFE=0.026642  MAE=0.019280  N=8
  post2  RMSFE=0.021223  MAE=0.016484  N=8
--- COVID      (2020-2021) ---
  m1  RMSFE=0.068916  MAE=0.042408  N=7
  m2  RMSFE=0.052123  MAE=0.038205  N=7
  m3  RMSFE=0.043052  MAE=0.025480  N=7
  post1  RMSFE=0.022938  MAE=0.016303  N=7
  post2  RMSFE=0.018235  MAE=0.015682  N=7
--- Post-COVID (2022-2025) ---
  m1  RMSFE=0.017171  MAE=0.014479  N=16
  m2  RMSFE=0.018126  MAE=0.013743  N=16
  m3  RMSFE=0.020921  MAE=0.017087  N=16
  post1  RMSFE=0.016393  MAE=0.013833  N=16
  post2  RMSFE=0.021421  MAE=0.019196  N=16
--- Full test  (2018-2025) ---
  m1  RMSFE=0.035193  MAE=0.019743  N=32
  m2  RMSFE=0.028236  MAE=0.018278  N=32
  m3  RMSFE=0.026471  MAE=0.017594  N=32
  post1  RMSFE=0.021240  MAE=0.016173  N=32
  post2  RMSFE=0.020396  MAE=0.017345  N=